In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [2]:
import matplotlib.pyplot as plt

In [3]:
from sklearn.decomposition import PCA

# Download_data

In [4]:
tahoe_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/tahoe/deg_data/group_rep/full/qc_false/filter_min_cells_50/results/'
tahoe_files = os.listdir(tahoe_path)

l1000_phase1_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase1/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase1_files = os.listdir(l1000_phase1_path)

l1000_phase2_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase2_files = os.listdir(l1000_phase2_path)

In [5]:
tahoe_overlap_l1000_phase1 = sorted(list(set(tahoe_files).intersection(set(l1000_phase1_files))))
tahoe_overlap_l1000_phase2 = sorted(list(set(tahoe_files).intersection(set(l1000_phase2_files))))

tahoe_overlap = sorted(set(tahoe_overlap_l1000_phase1 + tahoe_overlap_l1000_phase2))

In [6]:
tahoe_overlap_l1000_phase1

['CVCL_0023_de.h5ad',
 'CVCL_0320_de.h5ad',
 'CVCL_0332_de.h5ad',
 'CVCL_0399_de.h5ad',
 'CVCL_0504_de.h5ad',
 'CVCL_0546_de.h5ad']

In [7]:
tahoe_overlap_l1000_phase2

['CVCL_0023_de.h5ad', 'CVCL_0320_de.h5ad', 'CVCL_0332_de.h5ad']

In [8]:
tahoe = []
for file in tqdm(tahoe_overlap):
    tahoe.append(ad.read_h5ad(tahoe_path + file))

100%|██████████| 6/6 [01:43<00:00, 17.31s/it]


In [9]:
l1000_phase1 = []
for file in tqdm(tahoe_overlap_l1000_phase1):
    l1000_phase1.append(ad.read_h5ad(l1000_phase1_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
100%|██████████| 6/6 [00:23<00:00,  3.86s/it]


In [10]:
l1000_phase2 = []
for file in tqdm(tahoe_overlap_l1000_phase2):
    l1000_phase2.append(ad.read_h5ad(l1000_phase2_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
100%|██████████| 3/3 [00:08<00:00,  2.93s/it]


In [11]:
for i, tahoe_i in enumerate(tahoe):
    for j, l1000_phase1_j in enumerate(l1000_phase1):
        if tahoe_i.obs['cell_type'].unique()[0] == l1000_phase1_j.obs['cell_type'].unique()[0]:
            print(f'{i, j}, {len(set(tahoe_i.obs['pubchem_cid']).intersection(l1000_phase1_j.obs['pubchem_cid']))} overlapping compounds in {tahoe_i.obs['cell_type'].unique()[0]}')

(0, 0), 96 overlapping compounds in CVCL_0023
(1, 1), 95 overlapping compounds in CVCL_0320
(2, 2), 11 overlapping compounds in CVCL_0332
(3, 3), 19 overlapping compounds in CVCL_0399
(4, 4), 18 overlapping compounds in CVCL_0504
(5, 5), 18 overlapping compounds in CVCL_0546


In [12]:
for i, tahoe_i in enumerate(tahoe):
    for j, l1000_phase2_j in enumerate(l1000_phase2):
        if tahoe_i.obs['cell_type'].unique()[0] == l1000_phase2_j.obs['cell_type'].unique()[0]:
            print(f'{i, j}, {len(set(tahoe_i.obs['pubchem_cid']).intersection(l1000_phase2_j.obs['pubchem_cid']))} overlapping compounds in {tahoe_i.obs['cell_type'].unique()[0]}')

(0, 0), 24 overlapping compounds in CVCL_0023
(1, 1), 110 overlapping compounds in CVCL_0320
(2, 2), 9 overlapping compounds in CVCL_0332


# Processing

In [13]:
def match_by_cid_and_closest_log_dose(tahoe_adata, l1000_adata):
    """For each row in tahoe_adata, find the row in l1000_adata with the same
    pubchem_cid and the closest pert_dose_uM in log space.

    Returns a copy of tahoe_adata.obs with an added 'matched_l1000_idx' column
    containing the matched obs index from l1000_adata.
    """
    l1000_obs = l1000_adata.obs.copy()
    l1000_obs['log_dose'] = np.log(l1000_obs['pert_dose_uM'])

    tahoe_obs = tahoe_adata.obs.copy()
    tahoe_obs['log_dose'] = np.log(tahoe_obs['pert_dose_uM'])

    l1000_by_cid = {cid: grp for cid, grp in l1000_obs.groupby('pubchem_cid', observed=True)}

    matched_l1000_idx = []
    for _, row in tahoe_obs.iterrows():
        l1000_grp = l1000_by_cid[row['pubchem_cid']]
        closest = (l1000_grp['log_dose'] - row['log_dose']).abs().idxmin()
        matched_l1000_idx.append(closest)

    tahoe_obs['matched_l1000_idx'] = matched_l1000_idx
    return tahoe_obs, l1000_obs

def return_embeddings(adata, dim=64,):
    adata_obs = adata.obs.copy()
    pca = PCA(n_components=dim)
    emb_logFC = pca.fit_transform(adata.layers['logFC'])
    
    pca = PCA(n_components=dim)
    emb_t = pca.fit_transform(adata.layers['t'])
    
    adata_obs['PCA.logFC'] = emb_logFC.tolist()
    adata_obs['PCA.t'] = emb_t.tolist()
    return adata_obs

In [14]:
cols_l1000 = ['PCA.logFC', 
              'PCA.t', 
              'pert_dose_uM', 
              'pert_time_h']

In [15]:
rename_l1000 = {'pert_time_h': 'l1000_pert_time_h',
                'pert_dose_uM': 'l1000_pert_dose_uM'}

## Phase2

### CVCL_0320

In [16]:
t_id = 1
l_id = 1

In [17]:
compounds_overlapped = list(set(tahoe[t_id].obs['pubchem_cid']).intersection(l1000_phase2[l_id].obs['pubchem_cid']))

In [18]:
len(compounds_overlapped)

110

In [19]:
tahoe_filtered = tahoe[t_id][tahoe[t_id].obs['pubchem_cid'].isin(compounds_overlapped)]
tahoe_filtered = tahoe_filtered[tahoe_filtered.obs['pert_dose_uM'] == 5].copy()

In [20]:
l1000_phase2_no_duplicates = l1000_phase2[l_id][~l1000_phase2[l_id].obs.index.duplicated()]
l1000_phase2_filtered = l1000_phase2_no_duplicates[l1000_phase2_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]

In [21]:
tahoe_obs, l1000_phase2_filtered_obs = match_by_cid_and_closest_log_dose(tahoe_filtered, l1000_phase2_filtered)

### 64

In [22]:
#V1
l1000_phase2_v1 = l1000_phase2_filtered[l1000_phase2_filtered.obs.index.isin(tahoe_obs['matched_l1000_idx'])]
l1000_phase2_obs_v1_64 = return_embeddings(l1000_phase2_v1)
print('matched doses:', l1000_phase2_obs_v1_64['pert_dose_uM'].unique())
tahoe_emb_v1_64_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v1_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

matched doses: [3.33333 3.33   ]


In [23]:
#V2
l1000_phase2_v2 = l1000_phase2_no_duplicates.copy()
l1000_phase2_obs_v2_64 = return_embeddings(l1000_phase2_v2)
tahoe_emb_v2_64_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v2_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [24]:
#V3
l1000_phase2_v3 = l1000_phase2_no_duplicates[l1000_phase2_no_duplicates.obs['pert_dose_uM'].isin([3.33, 3.33333])].copy()
l1000_phase2_obs_v3_64 = return_embeddings(l1000_phase2_v3)
tahoe_emb_v3_64_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v3_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [25]:
# V4
l1000_phase2_v4 = l1000_phase2_filtered.copy()
l1000_phase2_obs_v4_64 = return_embeddings(l1000_phase2_v4)
tahoe_emb_v4_64_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v4_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

### 128

In [26]:
#V2
l1000_phase2_v2 = l1000_phase2_no_duplicates.copy()
l1000_phase2_obs_v2_128 = return_embeddings(l1000_phase2_v2, dim=128)
tahoe_emb_v2_128_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v2_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [27]:
#V3
l1000_phase2_v3 = l1000_phase2_no_duplicates[l1000_phase2_no_duplicates.obs['pert_dose_uM'].isin([3.33, 3.33333])].copy()
l1000_phase2_obs_v3_128 = return_embeddings(l1000_phase2_v3, dim=128)
tahoe_emb_v3_128_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v3_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [28]:
# V4
l1000_phase2_v4 = l1000_phase2_filtered.copy()
l1000_phase2_obs_v4_128 = return_embeddings(l1000_phase2_v4, dim=128)
tahoe_emb_v4_128_l1000_phase2 = tahoe_obs.merge(l1000_phase2_obs_v4_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

## Phase1

### CVCL_0023

In [29]:
t_id = 0
l_id = 0

In [30]:
# as tahoe has only 24 hour time
compounds_overlapped = list(set(tahoe[t_id].obs['pubchem_cid']).intersection(l1000_phase1[l_id][l1000_phase1[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))

In [31]:
len(compounds_overlapped)

83

In [32]:
tahoe_filtered = tahoe[t_id][tahoe[t_id].obs['pubchem_cid'].isin(compounds_overlapped)]
tahoe_filtered = tahoe_filtered[tahoe_filtered.obs['pert_dose_uM'] == 5].copy()

In [33]:
l1000_phase1_no_duplicates = l1000_phase1[l_id][~l1000_phase1[l_id].obs.index.duplicated()]
l1000_phase1_filtered = l1000_phase1_no_duplicates[l1000_phase1_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
l1000_phase1_filtered_time = l1000_phase1_filtered[l1000_phase1_filtered.obs['pert_time_h'] == 24].copy()

In [34]:
tahoe_obs, _ = match_by_cid_and_closest_log_dose(tahoe_filtered, l1000_phase1_filtered_time)

### 64

In [35]:
#V1
#filter by compounds and dose, prevailing dose is 10!
l1000_phase1_v1 = l1000_phase1_filtered_time[(l1000_phase1_filtered_time.obs.index.isin(tahoe_obs['matched_l1000_idx']))&(l1000_phase1_filtered_time.obs['pert_dose_uM'] == 10)]
l1000_phase1_obs_v1_64 = return_embeddings(l1000_phase1_v1)
print('matched doses:', l1000_phase1_obs_v1_64['pert_dose_uM'].unique())
tahoe_emb_v1_64_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v1_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

matched doses: [10.]


In [36]:
#V2
l1000_phase1_v2 = l1000_phase1_no_duplicates.copy()
l1000_phase1_obs_v2_64 = return_embeddings(l1000_phase1_v2)
l1000_phase1_obs_v2_64_filtered_dose = l1000_phase1_obs_v2_64[l1000_phase1_obs_v2_64['pert_dose_uM'] == 10].copy()
tahoe_emb_v2_64_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v2_64_filtered_dose[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [37]:
#V3
l1000_phase1_v3 = l1000_phase1_no_duplicates[(l1000_phase1_no_duplicates.obs['pert_dose_uM'].isin([10.]))&(l1000_phase1_no_duplicates.obs['pert_time_h'].isin([24.]))].copy()
l1000_phase1_obs_v3_64 = return_embeddings(l1000_phase1_v3)
tahoe_emb_v3_64_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v3_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [38]:
# V4
l1000_phase1_v4 = l1000_phase1_filtered.copy()
l1000_phase1_obs_v4_64 = return_embeddings(l1000_phase1_v4)
l1000_phase1_obs_v4_64_filtered_dose = l1000_phase1_obs_v4_64[l1000_phase1_obs_v4_64['pert_dose_uM'] == 10].copy()
tahoe_emb_v4_64_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v4_64_filtered_dose[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

### 128

In [39]:
#V2
l1000_phase1_v2 = l1000_phase1_no_duplicates.copy()
l1000_phase1_obs_v2_128 = return_embeddings(l1000_phase1_v2, dim=128)
l1000_phase1_obs_v2_128_filtered_dose = l1000_phase1_obs_v2_128[l1000_phase1_obs_v2_128['pert_dose_uM'] == 10].copy()

tahoe_emb_v2_128_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v2_128_filtered_dose[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

#V3
l1000_phase1_v3 = l1000_phase1_no_duplicates[(l1000_phase1_no_duplicates.obs['pert_dose_uM'].isin([10.]))&(l1000_phase1_no_duplicates.obs['pert_time_h'].isin([24.]))].copy()
l1000_phase1_obs_v3_128 = return_embeddings(l1000_phase1_v3, dim=128)
tahoe_emb_v3_128_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v3_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

# V4
l1000_phase1_v4 = l1000_phase1_filtered.copy()
l1000_phase1_obs_v4_128 = return_embeddings(l1000_phase1_v4, dim=128)
l1000_phase1_obs_v4_128_filtered_dose = l1000_phase1_obs_v4_128[l1000_phase1_obs_v4_128['pert_dose_uM'] == 10].copy()

tahoe_emb_v4_128_l1000_phase1 = tahoe_obs.merge(l1000_phase1_obs_v4_128_filtered_dose[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

### CVCL_0320

In [40]:
t_id = 1
l_id = 1

In [41]:
# as tahoe has only 24 hour time
compounds_overlapped = list(set(tahoe[t_id].obs['pubchem_cid']).intersection(l1000_phase1[l_id][l1000_phase1[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))

In [42]:
len(compounds_overlapped) # if we match by hours either we have only 24 compounds...

24

# Put all together

In [43]:
tahoe_emb_v1_64_l1000_phase2['version'] = 'v1'
tahoe_emb_v2_64_l1000_phase2['version'] = 'v2'
tahoe_emb_v3_64_l1000_phase2['version'] = 'v3'
tahoe_emb_v4_64_l1000_phase2['version'] = 'v4'

tahoe_emb_v2_128_l1000_phase2['version'] = 'v2'
tahoe_emb_v3_128_l1000_phase2['version'] = 'v3'
tahoe_emb_v4_128_l1000_phase2['version'] = 'v4'

tahoe_emb_v1_64_l1000_phase1['version'] = 'v1'
tahoe_emb_v2_64_l1000_phase1['version'] = 'v2'
tahoe_emb_v3_64_l1000_phase1['version'] = 'v3'
tahoe_emb_v4_64_l1000_phase1['version'] = 'v4'

tahoe_emb_v2_128_l1000_phase1['version'] = 'v2'
tahoe_emb_v3_128_l1000_phase1['version'] = 'v3'
tahoe_emb_v4_128_l1000_phase1['version'] = 'v4'

In [44]:
tahoe_emb_64_l1000_phase2 = pd.concat([tahoe_emb_v1_64_l1000_phase2,
           tahoe_emb_v2_64_l1000_phase2,
           tahoe_emb_v3_64_l1000_phase2,
           tahoe_emb_v4_64_l1000_phase2
          ])

In [45]:
tahoe_emb_128_l1000_phase2 = pd.concat([tahoe_emb_v2_128_l1000_phase2,
           tahoe_emb_v3_128_l1000_phase2,
           tahoe_emb_v4_128_l1000_phase2
          ])

In [46]:
tahoe_emb_64_l1000_phase1 = pd.concat([tahoe_emb_v1_64_l1000_phase1,
           tahoe_emb_v2_64_l1000_phase1,
           tahoe_emb_v3_64_l1000_phase1,
           tahoe_emb_v4_64_l1000_phase1
          ])

In [47]:
tahoe_emb_128_l1000_phase1 = pd.concat([tahoe_emb_v2_128_l1000_phase1,
           tahoe_emb_v3_128_l1000_phase1,
           tahoe_emb_v4_128_l1000_phase1
          ])

In [48]:
tahoe_emb_64_l1000_phase2['dim'] = 64
tahoe_emb_128_l1000_phase2['dim'] = 128

tahoe_emb_64_l1000_phase1['dim'] = 64
tahoe_emb_128_l1000_phase1['dim'] = 128

tahoe_emb_64_l1000_phase2['l1000_phase'] = 2
tahoe_emb_128_l1000_phase2['l1000_phase'] = 2

tahoe_emb_64_l1000_phase1['l1000_phase'] = 1
tahoe_emb_128_l1000_phase1['l1000_phase'] = 1

In [49]:
tahoe_emb = pd.concat([tahoe_emb_64_l1000_phase2, 
          tahoe_emb_128_l1000_phase2,
          tahoe_emb_64_l1000_phase1, 
          tahoe_emb_128_l1000_phase1])

In [50]:
tahoe_emb = tahoe_emb[~tahoe_emb['PCA.t'].isna()]

In [51]:
columns = ['cell_type', 'perturbagen', 'pert_type',
       'is_control', 'pert_dose_uM', 'pert_time_h',
        'pubchem_cid', 'perturbation_label', 'matched_l1000_idx',
        'l1000_pert_dose_uM', 'l1000_pert_time_h', 
        'PCA.logFC', 'PCA.t', 'version', 'dim',
       'l1000_phase']

In [52]:
tahoe_emb = tahoe_emb[columns].reset_index(drop=True).copy()

In [53]:
tahoe_emb[(tahoe_emb['cell_type'] == 'CVCL_0320')]['l1000_pert_dose_uM'].unique()

array([3.33   , 3.33333])

In [54]:
tahoe_sci_op3_updated = pd.read_pickle('../../tahoe_sci_op3_updated.pkl')

In [55]:
tahoe_emb = tahoe_emb.merge(tahoe_sci_op3_updated[tahoe_sci_op3_updated['dataset'] == 'tahoe'][['perturbagen','original_pert_name']], how='left', on='perturbagen')

In [56]:
tahoe_emb

,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,pubchem_cid,perturbation_label,matched_l1000_idx,l1000_pert_dose_uM,l1000_pert_time_h,PCA.logFC,PCA.t,version,dim,l1000_phase,original_pert_name
0,CVCL_0320,alpelisib,compound,FALSE,5.0,24.0,56649450,alpelisib_5uM_24h,56649450_3_33uM_24h - 679_24h,3.33000,24.0,"[-6.29152089212975, -2.4460581799616086, -1.50...","[-25.571593719917146, 10.967763797756612, 5.94...",v1,64,2,Alpelisib
1,CVCL_0320,palbociclib,compound,FALSE,5.0,24.0,5330286,palbociclib_5uM_24h,5330286_3_3333uM_24h - 679_24h,3.33333,24.0,"[-13.19793165437994, -0.08249463076798, 0.5354...","[-53.2878789000218, -3.669506747244565, 1.5673...",v1,64,2,palbociclib
2,CVCL_0320,Infigratinib,compound,FALSE,5.0,24.0,53235510,Infigratinib_5uM_24h,53235510_3_33uM_24h - 679_24h,3.33000,24.0,"[4.480231322135845, -1.1060817904088174, -3.19...","[14.759168445466964, 3.1262594216205586, 13.08...",v1,64,2,Infigratinib
3,CVCL_0320,capivasertib,compound,FALSE,5.0,24.0,25227436,capivasertib_5uM_24h,25227436_3_3333uM_24h - 679_24h,3.33333,24.0,"[1.9403453409102045, -2.5021069117108006, 0.55...","[8.874716345253495, 8.004876061976683, -1.1839...",v1,64,2,Capivasertib
4,CVCL_0320,irinotecan,compound,FALSE,5.0,24.0,60838,irinotecan_5uM_24h,60838_3_3333uM_24h - 679_24h,3.33333,24.0,"[1.5883934302821328, -1.0127581498046516, 2.91...","[7.131042477606337, 3.923362131855394, -7.7857...",v1,64,2,Irinotecan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1297,CVCL_0023,pimozide,compound,FALSE,5.0,24.0,16362,pimozide_5uM_24h,16362_10uM_24h - 679_24h,10.00000,24.0,"[-0.5108333021362381, -1.6709320754185015, -2....","[-2.2666823223152748, -5.863259267766386, 7.94...",v4,128,1,Pimozide
1298,CVCL_0023,gefitinib,compound,FALSE,5.0,24.0,123631,gefitinib_5uM_24h,123631_10uM_24h - 679_24h,10.00000,24.0,"[-4.435991623329821, -2.654278702691419, -1.40...","[-13.764374308952382, -11.255879690758508, 3.0...",v4,128,1,Gefitinib
1299,CVCL_0023,nimesulide,compound,FALSE,5.0,24.0,4495,nimesulide_5uM_24h,4495_10uM_24h - 679_24h,10.00000,24.0,"[0.7932697947000465, -0.9187910067682297, 0.32...","[4.343751859178172, -2.399327009425929, -2.933...",v4,128,1,Nimesulide
1300,CVCL_0023,Carbidopa (monohydrate),compound,FALSE,5.0,24.0,2563,Carbidopa (monohydrate)_5uM_24h,2563_10uM_24h - 679_24h,10.00000,24.0,"[-6.858054314866675, -2.390621724536526, 0.307...","[-10.140098574074862, -6.425200375933785, -10....",v4,128,1,Carbidopa (monohydrate)


In [57]:
tahoe_emb.to_pickle('tahoe_PCA_emb.pkl')